# Week 8 Capstone: RFC Spec Assistant

A developer asks a question about JSON, timestamps, Base64, JWT, OAuth bearer tokens, cookies, WebSocket, or UUIDs. Claude answers from eight IETF RFCs using two tools: `search_rfcs` (retrieval with section-aware chunks and cross-encoder re-ranking) and `run_python` (a guarded arithmetic evaluator). A versioned system prompt requires grounded, cited answers, tool-computed numbers, and an explicit refusal when the corpus does not cover the question. Gemini, a different model family, judges the answers.

```
question ─► Claude + prompts/system_v1.txt ─┬─► search_rfcs(query) ─► FAISS top-20 (section chunks) ─► cross-encoder ─► top-3 [RFC n] passages
                                            └─► run_python(expr)   ─► 2-stage AST guard ─► number
            ◄──────────── tool results (logged per call) ────────────┘
answer ─► model-free metrics + Gemini judge (prompts/judge_v1.txt)
```

| Technique | Reused from |
|---|---|
| Corpus, section-aware chunking, FAISS search | Week 5, `05-rag-and-retrieval/week5_rag_pipeline.ipynb` |
| Cross-encoder re-ranking (N=20 → K=3) | Week 6, `06-advanced-rag-systems/week6_advanced_rag.ipynb` |
| Tool schemas, guarded `run_python`, `validate`/`dispatch`, tool loop | Week 4, `04-function-and-tool-use/week4_tool_use.ipynb` |
| Versioned prompt files | Week 3, `03-prompt-engineering/week3_prompt_engineering.ipynb` |
| Throttled, cached Gemini calls; cross-family judge | Week 7 (`week7_adaptation`) and Week 6 (`diagnosing_rag_pipe.ipynb`) |

## Setup
Generator runs are cached in `eval_runs.json` and judge calls in `gemini_cache.json`, so re-running the notebook makes no new API calls for work already done.

In [1]:
import os, re, ast, json, math, time, string, pathlib, threading, urllib.request, numpy as np
os.environ['HF_HOME'] = str((pathlib.Path('.') / '.hf_cache').resolve())
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss

load_dotenv()
ANTHROPIC_OK, GEMINI_OK = bool(os.getenv('ANTHROPIC_API_KEY')), bool(os.getenv('GEMINI_API_KEY'))
print('ANTHROPIC_API_KEY set:', ANTHROPIC_OK, '| GEMINI_API_KEY set:', GEMINI_OK)

CLAUDE_MODEL = 'claude-haiku-5-5'      # generator
JUDGE_MODEL = 'gemini-3.1-flash-lite'  # judge
PROMPT_DIR = pathlib.Path('prompts')

ANTHROPIC_API_KEY set: True | GEMINI_API_KEY set: True


## Component 1: Retrieval (Weeks 5 and 6)

From Week 5: the RFC loader, paragraph chunker, and the heading-aware `chunk_sections` a mitigation for headings retrieved as empty chunks. 

From Week 6: the cross-encoder `rerank`, which was the only transform that improved ranking without regressing any query 

In [2]:
RFCS = {8259: 'JSON', 3339: 'Date and time', 4648: 'Base16/32/64 encodings', 7519: 'JSON Web Token',
        6750: 'OAuth 2.0 bearer tokens', 6265: 'HTTP cookies', 6455: 'WebSocket', 9562: 'UUIDs'}
RFC_DIR = pathlib.Path('../06-advanced-rag-systems/rfcs')

def load_rfc(n):
    path = RFC_DIR / f'rfc{n}.txt'
    if not path.exists():
        urllib.request.urlretrieve(f'https://www.rfc-editor.org/rfc/rfc{n}.txt', path)
    text = path.read_text(encoding='utf-8')
    # Page break = footer line, form feed, running header (absent after the last page).
    return re.sub(r'\n[^\n]*\[Page \d+\]\n\f\n(?:RFC \d+[^\n]*\n)?', '\n', text)

CORPUS = {n: load_rfc(n) for n in RFCS}

def chunk_paragraph(text):
    return [re.sub(r'\s+',' ',p).strip() for p in text.split('\n\n') if p.strip()]
def chunk_corpus(chunker, *args):
    # Chunk each RFC separately so no chunk spans two documents; tag each chunk with its RFC.
    return [(n, c) for n, text in CORPUS.items() for c in chunker(text, *args)]

HEADING = re.compile(r'^(?:\d+(?:\.\d+)*\.?|Appendix [A-Z]\.?)\s+\S.{0,70}$')
TOC = re.compile(r'(?:\. ?){4,}')  # dot leaders: '....' or '. . . .'
def chunk_sections(text):
    out, heading = [], ''
    for p in chunk_paragraph(text):
        if TOC.search(p):
            continue
        if HEADING.match(p):
            heading = p; continue
        out.append(f'{heading}: {p}' if heading else p)
    return out

embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L6-v2')

def build(chunks):
    e = embedder.encode([t for _, t in chunks], normalize_embeddings=True).astype('float32')
    idx = faiss.IndexFlatIP(e.shape[1]); idx.add(e); return idx
def search(idx, q, k=3):
    qe = embedder.encode([q], normalize_embeddings=True).astype('float32')
    # FAISS uses -1 for missing results when fewer than k chunks exist.
    return [i for i in idx.search(qe, k)[1][0] if i >= 0]

CHUNKS = {'paragraph': chunk_corpus(chunk_paragraph), 'sections': chunk_corpus(chunk_sections)}
INDEX = {name: build(c) for name, c in CHUNKS.items()}

N, K = 20, 3
def rerank(q, chunks, cand):
    scores = reranker.predict([(q, chunks[i][1]) for i in cand])
    return [cand[j] for j in np.argsort(scores)[::-1]]

def retrieve_ids(q, chunker='sections', rerank_on=True, k=K):
    cand = search(INDEX[chunker], q, N if rerank_on else k)
    return (rerank(q, CHUNKS[chunker], cand) if rerank_on else cand)[:k]
def retrieve(q, chunker='sections', rerank_on=True, k=K):
    return [CHUNKS[chunker][i] for i in retrieve_ids(q, chunker, rerank_on, k)]

for name, c in CHUNKS.items():
    print(f'{name:10s} -> {len(c):5d} chunks')

paragraph  ->  2604 chunks
sections   ->  2228 chunks


**Component check.** 
Before integrating, the retrieval component is re-scored on Week 6's 10 labelled queries for all four combinations of chunker and re-ranking. 

In [3]:
# (query, relevant RFC, acceptable answer strings), copied from Week 6
qa = [
    ('what text encoding must JSON use when exchanged between systems', 8259,
     ('MUST be encoded using UTF-8', 'require the use of UTF-8 when transmitted')),
    ('what character separates the date from the time in an internet timestamp', 3339,
     ('full-date "T" full-time', 'date and time separated by "T"', 'requires the "T"')),
    ('which character replaces the plus sign in URL-safe base64', 4648,
     ('62 - (minus)',)),
    ('which claim stops a JWT from being used before a certain time', 7519,
     ('the time before which the JWT MUST NOT be accepted',)),
    ('what status code should a server return for an expired access token', 6750,
     ('HTTP 401 (Unauthorized)', 'expired access token: HTTP/1.1 401')),
    ('response when the token lacks the permissions a request needs', 6750,
     ('403 (Forbidden)',)),
    ('if a cookie sets both Max-Age and Expires, which one wins', 6265,
     ('attribute has precedence',)),
    ('which opcode is used for a websocket ping frame', 6455,
     ('%x9 denotes a ping', '%x9 ; ping', '0x9 (Ping)', 'opcode of 0x9')),
    ('what protocol version does the client send in the websocket opening handshake', 6455,
     ('with a value of 13', 'MUST be 13', 'Sec-WebSocket-Version: 13')),
    ('how many bits of a UUIDv7 hold the millisecond timestamp', 9562,
     ('most significant 48 bits', '48-bit big-endian unsigned number of the Unix Epoch',
      'UUIDv7 features a 48-bit timestamp', 'millisecond precision to fill the first 48 bits')),
]
norm = lambda s: re.sub(r'\s+', ' ', s).lower()

def relevant(chunks, rfc, facts):
    return {i for i, (n, t) in enumerate(chunks) if n == rfc and any(norm(f) in norm(t) for f in facts)}
def first_rank(ids, rel):
    return next((r for r, i in enumerate(ids, 1) if i in rel), None)

print(f'{"chunker":10s} {"rerank":>6s} {"hit@3":>6s} {"MRR@3":>6s}')
for chunker, chunks in CHUNKS.items():
    labels = [relevant(chunks, rfc, facts) for _, rfc, facts in qa]
    for rerank_on in (False, True):
        ranks = [first_rank(retrieve_ids(q, chunker, rerank_on), rel) for (q, _, _), rel in zip(qa, labels)]
        print(f'{chunker:10s} {str(rerank_on):>6s} {np.mean([r is not None for r in ranks]):6.2f} '
              f'{np.mean([1 / r if r else 0 for r in ranks]):6.2f}')

chunker    rerank  hit@3  MRR@3
paragraph   False   0.70   0.55


paragraph    True   0.80   0.80
sections    False   0.70   0.48


sections     True   0.80   0.80


Both chunkers reach hit@3 0.80 once re-ranked; re-ranking is what moves MRR@3 (0.55 → 0.80 for paragraphs, 0.48 → 0.80 for sections). Section chunks are kept because every passage carries its section heading, which gives the generator a topic label and a citation anchor.

## Component 2: Tools (Week 4)

From Week 4: the two-stage guarded evaluator behind `run_python`, the schema validator, and `dispatch`, which returns errors to the model instead of raising. 

In [4]:
class GuardError(Exception):
    """Input rejected by the guard, before any evaluation."""

MAX_EXPR_CHARS = 200   # matches maxLength in the schema
MAX_POW_EXPONENT = 64  # bounds the cost of a single ** operation
TIME_LIMIT_S = 2.0     # backstop for anything the structural caps miss

# The allowlist. Every node in the parsed tree must be an instance of one of these.
_ALLOWED_NODES = (
    ast.Expression,
    ast.Constant,
    ast.BinOp, ast.UnaryOp,
    ast.Add, ast.Sub, ast.Mult, ast.Div, ast.FloorDiv, ast.Mod, ast.Pow,
    ast.USub, ast.UAdd,
)
_OPS = {
    ast.Add: lambda a, b: a + b,
    ast.Sub: lambda a, b: a - b,
    ast.Mult: lambda a, b: a * b,
    ast.Div: lambda a, b: a / b,
    ast.FloorDiv: lambda a, b: a // b,
    ast.Mod: lambda a, b: a % b,
    ast.Pow: lambda a, b: a ** b,
}
_UNARY = {ast.USub: lambda a: -a, ast.UAdd: lambda a: +a}

def _literal_number(node):
    """Return the value of a numeric literal, or None if the node is anything else."""
    if isinstance(node, ast.Constant) and type(node.value) in (int, float):
        return node.value
    if isinstance(node, ast.UnaryOp) and isinstance(node.op, (ast.USub, ast.UAdd)):
        inner = _literal_number(node.operand)
        if inner is None:
            return None
        return -inner if isinstance(node.op, ast.USub) else inner
    return None

def _approve(tree):
    """Stage 1: walk the entire tree and reject anything not on the allowlist."""
    for node in ast.walk(tree):
        if not isinstance(node, _ALLOWED_NODES):
            raise GuardError(f"disallowed syntax: {type(node).__name__}")
        if isinstance(node, ast.Constant) and type(node.value) not in (int, float):
            raise GuardError(f"only int and float literals are allowed, got {type(node.value).__name__}")
        if isinstance(node, ast.BinOp) and isinstance(node.op, ast.Pow):
            exponent = _literal_number(node.right)
            if exponent is None:
                raise GuardError("** exponent must be a numeric literal, not a computed value")
            if abs(exponent) > MAX_POW_EXPONENT:
                raise GuardError(f"** exponent magnitude exceeds {MAX_POW_EXPONENT}")

def _evaluate(node):
    """Stage 2: evaluate an already-approved tree."""
    if isinstance(node, ast.Constant):
        return node.value
    if isinstance(node, ast.BinOp):
        return _OPS[type(node.op)](_evaluate(node.left), _evaluate(node.right))
    if isinstance(node, ast.UnaryOp):
        return _UNARY[type(node.op)](_evaluate(node.operand))
    raise GuardError(f"unreachable: {type(node).__name__} passed approval")

def _evaluate_with_time_limit(tree, seconds):
    """Bounds how long the caller waits; a Python thread cannot be killed from outside."""
    box = {}
    def worker():
        try:
            box["value"] = _evaluate(tree.body)
        except BaseException as exc:  # re-raised on the calling thread below
            box["error"] = exc
    thread = threading.Thread(target=worker, daemon=True)
    thread.start()
    thread.join(seconds)
    if thread.is_alive():
        raise GuardError(f"evaluation exceeded the {seconds}s time limit")
    if "error" in box:
        raise box["error"]
    return box["value"]

def run_python(expression):
    if len(expression) > MAX_EXPR_CHARS:
        raise GuardError(f"expression exceeds {MAX_EXPR_CHARS} characters")
    try:
        tree = ast.parse(expression, mode="eval")
    except SyntaxError as exc:
        raise GuardError(f"could not parse expression: {exc.msg}")
    _approve(tree)
    result = _evaluate_with_time_limit(tree, TIME_LIMIT_S)
    if type(result) not in (int, float):  # e.g. (-8) ** 0.5 returns complex
        raise GuardError(f"result is not a real number: {type(result).__name__}")
    if isinstance(result, float) and not math.isfinite(result):
        raise GuardError("result is not finite")
    return result

In [5]:
TOOLS = [
    {
        "name": "search_rfcs",
        "description": (
            "Search eight IETF RFCs (JSON 8259, timestamps 3339, Base16/32/64 4648, JWT 7519, OAuth bearer "
            "tokens 6750, HTTP cookies 6265, WebSocket 6455, UUIDs 9562) and return the 3 most relevant "
            "passages, each tagged with its RFC number. Write the query as a short description of the fact "
            "you need. Search again with a different query if the passages do not contain it."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "maxLength": 300,
                    "description": "What to look up, e.g. 'websocket opcode for a ping frame'.",
                },
            },
            "required": ["query"],
            "additionalProperties": False,
        },
    },
    {
        "name": "run_python",
        "description": (
            "Evaluate a single arithmetic expression and return the result. "
            "Variables, function calls, imports, and attribute access are rejected so substitute concrete numbers in to the expression before calling."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "maxLength": 200,
                    "description": "Arithmetic over numeric literals, e.g. '5027.1 * 2.23694'. No names or calls.",
                },
            },
            "required": ["expression"],
            "additionalProperties": False,
        },
    },
]

def search_rfcs(query):
    return [{"rfc": n, "text": t} for n, t in retrieve(query)]

IMPL = {"search_rfcs": search_rfcs, "run_python": run_python}

class ToolArgError(Exception):
    """Arguments rejected against the schema, before the implementation is called."""

def validate(name, args):
    spec = next((t["input_schema"] for t in TOOLS if t["name"] == name), None)
    if spec is None:
        raise ToolArgError(f"unknown tool: {name}")
    if not isinstance(args, dict):
        raise ToolArgError("arguments must be a JSON object")
    for field in spec.get("required", []):
        if field not in args:
            raise ToolArgError(f"missing required field: {field}")
    for key, value in args.items():
        prop = spec["properties"].get(key)
        if prop is None:
            raise ToolArgError(f"unexpected field: {key} (expected one of {list(spec['properties'])})")
        if prop["type"] == "string":
            if not isinstance(value, str):
                raise ToolArgError(f"{key} must be a string, got {type(value).__name__}")
            if "maxLength" in prop and len(value) > prop["maxLength"]:
                raise ToolArgError(f"{key} exceeds maxLength {prop['maxLength']}")
        else:
            raise ValueError(f"extend validate() to support schema type: {prop['type']}")

def dispatch(name, args):
    """Validate, execute, and return a JSON-serializable result the model can read."""
    try:
        validate(name, args)
        output = IMPL[name](**args)
        json.dumps(output, allow_nan=False)  # reject results the model cannot receive as JSON
        return {"ok": True, "tool": name, "output": output}
    except ToolArgError as exc:
        return {"ok": False, "tool": name, "error_type": "invalid_arguments", "message": str(exc)}
    except GuardError as exc:
        return {"ok": False, "tool": name, "error_type": "guard_rejected", "message": str(exc)}
    except Exception as exc:
        return {"ok": False, "tool": name, "error_type": "execution_error", "message": str(exc)}

for p in dispatch("search_rfcs", {"query": "websocket opcode for a ping frame"})["output"]:
    print(f"[RFC {p['rfc']}] {p['text'][:100]}")
print(dispatch("run_python", {"expression": "(1000 + 2) // 3 * 4"}))
print(dispatch("run_python", {"expression": "round(1336 / 3)"}))

[RFC 6455] 5.5.2. Ping: The Ping frame contains an opcode of 0x9.
[RFC 6455] 11.8. WebSocket Opcode Registry: |Opcode | Meaning | Reference | -+--------+------------------------
[RFC 6455] 11.8. WebSocket Opcode Registry: Opcode The opcode denotes the frame type of the WebSocket frame, as
{'ok': True, 'tool': 'run_python', 'output': 1336}
{'ok': False, 'tool': 'run_python', 'error_type': 'guard_rejected', 'message': 'disallowed syntax: Call'}


## Integration: the tool loop (Week 4) with a versioned system prompt (Week 3)
Week 4's `run_conversation` with two changes: it takes a `system` prompt, and it returns its call log instead of appending to a global. The system prompt lives in `prompts/system_v1.txt` so a revision is a new file and runs under each version are cached separately.

In [6]:
from anthropic import Anthropic
claude = Anthropic()

def system_prompt(version):
    return (PROMPT_DIR / f'system_{version}.txt').read_text(encoding='utf-8')

def run_conversation(query, system, max_turns=6):
    """Run the tool loop until Claude answers in prose; return (answer, call log)."""
    messages = [{"role": "user", "content": query}]
    calls, turn = [], 0
    create = lambda: claude.messages.create(model=CLAUDE_MODEL, max_tokens=1024, system=system,
                                            tools=TOOLS, messages=messages)
    response = create()
    while response.stop_reason == "tool_use" and turn < max_turns:
        messages.append({"role": "assistant", "content": response.content})
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                result = dispatch(block.name, block.input)
                calls.append({"turn": turn, "tool": block.name, "args": block.input, "result": result})
                tool_results.append({"type": "tool_result", "tool_use_id": block.id,
                                     "content": json.dumps(result), "is_error": not result["ok"]})
        messages.append({"role": "user", "content": tool_results})
        response = create()
        turn += 1
    text = " ".join(b.text for b in response.content if b.type == "text").strip()
    return text or f"<no text returned: stop_reason={response.stop_reason}, turns={turn}>", calls

RUNS = pathlib.Path('eval_runs.json')
runs = json.loads(RUNS.read_text(encoding='utf-8')) if RUNS.exists() else {}

def ask(q, version='v1', trial=0):
    key = f'{CLAUDE_MODEL}|{version}|{q}' + (f'|trial {trial}' if trial else '')
    if key not in runs:
        answer, calls = run_conversation(q, system_prompt(version))
        runs[key] = {'q': q, 'version': version, 'answer': answer, 'calls': calls}
        RUNS.write_text(json.dumps(runs, indent=1), encoding='utf-8')
    return runs[key]

def contexts(run):
    return [p for c in run['calls'] if c['tool'] == 'search_rfcs' and c['result']['ok'] for p in c['result']['output']]

def show_run(run):
    print('>', run['q'])
    for c in run['calls']:
        r = c['result']
        out = r['message'] if not r['ok'] else (f"{len(r['output'])} passages from RFC {sorted({p['rfc'] for p in r['output']})}"
                                                if c['tool'] == 'search_rfcs' else r['output'])
        print(f"  [{'OK ' if r['ok'] else 'ERR'}] turn {c['turn']}: {c['tool']}({json.dumps(c['args'])}) -> {out}")
    print('  answer:', run['answer'], '\n')

show_run(ask('what character separates the date from the time in an internet timestamp'))
show_run(ask('how many characters does base64 produce for a 300-byte input'))

> what character separates the date from the time in an internet timestamp
  [OK ] turn 0: search_rfcs({"query": "RFC 3339 date and time separator character in internet date/time format"}) -> 3 passages from RFC [3339, 7519]
  answer: In RFC 3339 the date and time are separated by the uppercase letter "T" (taken from ISO 8601). For readability, applications may instead use a space character to separate a full-date and full-time [RFC 3339]. 

> how many characters does base64 produce for a 300-byte input
  [OK ] turn 0: search_rfcs({"query": "Base64 encoding output length with padding characters for input octets"}) -> 3 passages from RFC [4648]
  [OK ] turn 1: run_python({"expression": "300 / 3 * 4"}) -> 400.0
  answer: Base64 produces 400 characters for a 300-byte input, with no "=" padding. The input is an exact multiple of 24 bits (3 bytes), so the output is an integral multiple of 4 characters [RFC 4648]. 



## Part 2: Evaluate
**Held-out set (15 cases)**:
- **rag (8):** one fact per RFC, answerable from a single passage.
- **compute (4):** need a retrieved spec value and arithmetic: Base64 output length, the largest 16-bit WebSocket payload length, minimum per-domain cookie storage, and the UUIDv7 timestamp range in years.
- **out-of-scope (3):** not covered by the corpus; the correct behaviour is the exact refusal from the system prompt.

Each case has `facts` (strings that must appear in the retrieved passages, checked against the corpus with Week 5's no-missing/no-leak assert; `|` separates acceptable wordings of one fact), an `answer` pattern for the key value, the tools it should use, and a reference answer for the judge.

**Model-free metrics:** tool selection, retrieval hit (every fact string is in some passage the model received), answer match (key value pattern in the answer, commas removed), and tool calls per question.

In [7]:
REFUSAL = 'not covered by the indexed RFCs'
RAG, CALC = ('search_rfcs',), ('search_rfcs', 'run_python')
evalset = [
    dict(id=1, kind='rag', rfc=7519, tools=RAG, answer=r'\bexp\b',
         q='Which JWT claim marks the time on or after which the token must not be accepted?',
         facts=('expiration time on or after which',),
         ref='The "exp" (expiration time) claim; the JWT MUST NOT be accepted on or after that time.'),
    dict(id=2, kind='rag', rfc=6455, tools=RAG, answer=r'1009',
         q='What close status code does a WebSocket endpoint use when it receives a message too big to process?',
         facts=('1009 indicates',),
         ref='Status code 1009: the endpoint is terminating the connection because a message was too big to process.'),
    dict(id=3, kind='rag', rfc=6265, tools=RAG, answer=r'httponly',
         q='Which cookie attribute keeps a cookie away from JavaScript and other non-HTTP APIs?',
         facts=('limits the scope of the cookie to HTTP requests',),
         ref='The HttpOnly attribute limits the cookie to HTTP requests, so the user agent omits it from non-HTTP APIs such as scripts.'),
    dict(id=4, kind='rag', rfc=8259, tools=RAG, answer=r'unique',
         q='Are duplicate member names allowed in a JSON object?',
         facts=('names within an object SHOULD be unique',),
         ref='The names within an object SHOULD be unique; duplicates are not forbidden, but behaviour with them is unpredictable.'),
    dict(id=5, kind='rag', rfc=6750, tools=RAG, answer=r'authorization.{0,80}bearer|bearer.{0,80}authorization',
         q='How should a client send an OAuth bearer token in an HTTP request header?',
         facts=('Authorization: Bearer|using the "Authorization" request header field',),
         ref='In the Authorization request header using the Bearer scheme: "Authorization: Bearer <token>".'),
    dict(id=6, kind='rag', rfc=3339, tools=RAG, answer=r'leap second',
         q='Can the seconds field of an RFC 3339 timestamp ever be 60?',
         facts=('00-60|may have the value "60"',),
         ref='Yes. time-second may be 60 (00-60) depending on leap second rules.'),
    dict(id=7, kind='rag', rfc=9562, tools=RAG, answer=r'0{8}-0{4}-0{4}-0{4}-0{12}|zero',
         q='What is the value of the Nil UUID?',
         facts=('all 128 bits set to zero',),
         ref='All 128 bits set to zero: 00000000-0000-0000-0000-000000000000.'),
    dict(id=8, kind='rag', rfc=4648, tools=RAG, answer=r'==|two .{0,12}(=|pad)',
         q='When base64 encoding, what padding is added if the final input quantum is only 8 bits?',
         facts=('exactly 8 bits; here, the final unit of encoded output will be two characters followed by two',),
         ref='The final unit is two characters followed by two "=" padding characters.'),
    dict(id=9, kind='compute', rfc=4648, tools=CALC, answer=r'1336',
         q='How many characters long is the base64 encoding of a 1000-byte input, including padding?',
         facts=('24-bit groups of input bits as output strings of 4 encoded characters'
                '|exactly 8 bits; here, the final unit of encoded output will be two characters followed by two',),
         ref='1336 characters: each 3-byte group becomes 4 characters, and 1000 bytes is 334 groups (the last padded), so 334 * 4 = 1336.'),
    dict(id=10, kind='compute', rfc=6455, tools=CALC, answer=r'65535',
         q='What is the largest WebSocket payload length that can be expressed with the 16-bit extended payload length field?',
         facts=('16-bit unsigned integer are the payload length',),
         ref='65535 bytes, the maximum value of a 16-bit unsigned integer (2**16 - 1).'),
    dict(id=11, kind='compute', rfc=6265, tools=CALC, answer=r'204800',
         q='At minimum, how many bytes of cookies for a single domain should a general-use user agent be able to store?',
         facts=('At least 4096 bytes per cookie', 'At least 50 cookies per domain'),
         ref='204800 bytes: at least 50 cookies per domain times at least 4096 bytes per cookie.'),
    dict(id=12, kind='compute', rfc=9562, tools=CALC, answer=r'89(19|20|25|26)|108(89|95)',
         q='Roughly how many years after 1970 can the UUIDv7 millisecond timestamp represent before it overflows?',
         facts=('48-bit big-endian unsigned number of the Unix Epoch',),
         ref='About 8,900 years: unix_ts_ms is 48 bits of milliseconds, and 2**48 ms is roughly 8,919 years (until about the year 10889).'),
    dict(id=13, kind='out-of-scope', rfc=None, tools=RAG, answer=REFUSAL,
         q='What is the default TCP port for a Redis server?', facts=(),
         ref='Out of scope: the indexed RFCs do not cover this, so the assistant should say so instead of answering.'),
    dict(id=14, kind='out-of-scope', rfc=None, tools=RAG, answer=REFUSAL,
         q='How does HTTP/2 HPACK encode integers in header blocks?', facts=(),
         ref='Out of scope: the indexed RFCs do not cover this, so the assistant should say so instead of answering.'),
    dict(id=15, kind='out-of-scope', rfc=None, tools=RAG, answer=REFUSAL,
         q='What is the maximum size of a single TLS 1.3 record?', facts=(),
         ref='Out of scope: the indexed RFCs do not cover this, so the assistant should say so instead of answering.'),
]
wordings = lambda e: [w for f in e['facts'] for w in f.split('|')]
missing = [w for e in evalset for w in wordings(e) if norm(w) not in norm(CORPUS[e['rfc']])]
leaks = [(w, d) for e in evalset for w in wordings(e) for d in CORPUS if d != e['rfc'] and norm(w) in norm(CORPUS[d])]
overlap = {norm(e['q']) for e in evalset} & {norm(q) for q, _, _ in qa}
assert not missing and not leaks and not overlap, (missing, leaks, overlap)
print(len(evalset), 'cases; every fact string found only in its own RFC; no overlap with the tuning queries')

15 cases; every fact string found only in its own RFC; no overlap with the tuning queries


In [8]:
t0 = time.monotonic()
results = {e['id']: ask(e['q']) for e in evalset}
print(f'{len(results)} runs ready in {time.monotonic() - t0:.0f}s (cached runs are instant)')

15 runs ready in 0s (cached runs are instant)


In [9]:
def tools_called(run):
    return {c['tool'] for c in run['calls']}

def facts_in(e, passages):
    ctx = norm(' '.join(p['text'] for p in passages))
    return all(any(norm(w) in ctx for w in f.split('|')) for f in e['facts'])

def free_metrics(e, run):
    return {'tools': set(e['tools']) <= tools_called(run),
            'retrieval': facts_in(e, contexts(run)) if e['facts'] else None,
            'answer': bool(re.search(e['answer'], run['answer'].replace(',', ''), re.I)),
            'calls': len(run['calls'])}

KINDS = ('rag', 'compute', 'out-of-scope', None)
fmt = lambda v: '-' if v is None else ('ok' if v is True else 'MISS')

def print_free(results, free):
    print(f'{"id":>2s} {"kind":12s} {"tools called":24s} {"tools":>5s} {"retr":>5s} {"answer":>6s} {"calls":>5s}')
    for e in evalset:
        m = free[e['id']]
        called = ','.join(sorted(tools_called(results[e['id']]))) or '(none)'
        print(f'{e["id"]:2d} {e["kind"]:12s} {called:24s} {fmt(m["tools"]):>5s} {fmt(m["retrieval"]):>5s} {fmt(m["answer"]):>6s} {m["calls"]:5d}')
    print(f'\n{"kind":12s} {"n":>3s} {"tools":>6s} {"retr":>6s} {"answer":>7s} {"calls":>6s}')
    for kind in KINDS:
        rows = [free[e['id']] for e in evalset if kind is None or e['kind'] == kind]
        retr = [r['retrieval'] for r in rows if r['retrieval'] is not None]
        print(f'{kind or "all":12s} {len(rows):3d} {np.mean([r["tools"] for r in rows]):6.0%} '
              f'{(f"{np.mean(retr):6.0%}" if retr else "     -")} {np.mean([r["answer"] for r in rows]):7.0%} '
              f'{np.mean([r["calls"] for r in rows]):6.1f}')

free = {e['id']: free_metrics(e, results[e['id']]) for e in evalset}
print_free(results, free)

id kind         tools called             tools  retr answer calls
 1 rag          search_rfcs                 ok    ok     ok     1
 2 rag          search_rfcs                 ok    ok     ok     2
 3 rag          search_rfcs                 ok    ok     ok     1
 4 rag          search_rfcs                 ok    ok     ok     2
 5 rag          search_rfcs                 ok    ok     ok     1
 6 rag          search_rfcs                 ok    ok     ok     1
 7 rag          search_rfcs                 ok    ok     ok     1
 8 rag          search_rfcs                 ok    ok     ok     1
 9 compute      run_python,search_rfcs      ok    ok     ok     2
10 compute      run_python,search_rfcs      ok    ok     ok     2
11 compute      search_rfcs               MISS    ok   MISS     3
12 compute      run_python,search_rfcs      ok    ok     ok     2
13 out-of-scope search_rfcs                 ok     -     ok     1
14 out-of-scope search_rfcs                 ok     -     ok     2
15 out-of-

### LLM-as-judge
Gemini grades each answer 1 to 5 on three dimensions, using the rubric in `prompts/judge_v1.txt`:
- **correctness:** agrees with the reference answer.
- **grounding:** every claim is supported by the passages or tool results the generator actually received.
- **citation:** cites the RFC each fact comes from, correctly.

The judge sees the same passages and tool results as the generator, so grounding is graded against the evidence the generator had, not the whole corpus. `gemini_label` is Week 7's throttled, cached caller (temperature 0, free-tier pacing, retry on per-minute limits).

In [10]:
MIN_INTERVAL = 60 / 14                       # free tier allows 15 requests/min; stay just under
CACHE = pathlib.Path('gemini_cache.json')    # completed calls survive a crash/rerun
cache = json.loads(CACHE.read_text(encoding='utf-8')) if CACHE.exists() else {}
_last = 0.0
if GEMINI_OK:
    from openai import OpenAI, RateLimitError
    gemini = OpenAI(api_key=os.environ['GEMINI_API_KEY'], base_url='https://generativelanguage.googleapis.com/v1beta/openai/')

def gemini_label(prompt, model=JUDGE_MODEL, retries=6):
    global _last
    key = f'{model}\n{prompt}'
    if key in cache: return cache[key]
    for attempt in range(retries):
        time.sleep(max(0, MIN_INTERVAL - (time.time() - _last)))   # client-side throttle
        _last = time.time()
        try:
            out = gemini.chat.completions.create(model=model, temperature=0,
                    messages=[{'role': 'user', 'content': prompt}]).choices[0].message.content.strip()
        except RateLimitError as e:
            if 'PerDay' in str(e): raise                       # daily quota: retrying won't help
            m = re.search(r'retry in ([\d.]+)s', str(e))       # server-suggested delay, else backoff
            time.sleep((float(m.group(1)) if m else 2**attempt) + 1)
            continue
        cache[key] = out; CACHE.write_text(json.dumps(cache), encoding='utf-8')
        return out
    raise RuntimeError(f'still rate limited after {retries} retries')

JUDGE_PROMPT = string.Template((PROMPT_DIR / 'judge_v1.txt').read_text(encoding='utf-8'))
DIMS = ['correctness', 'grounding', 'citation']

def judge(e, run):
    ctx = '\n'.join(f"- [RFC {p['rfc']}] {p['text']}" for p in contexts(run)) or '(none)'
    tools = '\n'.join(f"- {c['args'].get('expression')} = {c['result'].get('output', c['result'].get('message'))}"
                      for c in run['calls'] if c['tool'] == 'run_python') or '(none)'
    out = gemini_label(JUDGE_PROMPT.substitute(question=e['q'], reference=e['ref'], contexts=ctx,
                                               tools=tools, answer=run['answer']))
    return json.loads(out.strip().removeprefix('```json').removeprefix('```').removesuffix('```').strip())

In [11]:
def print_judge(verdicts, free):
    print(f'{"id":>2s} {"kind":12s}' + ''.join(f'{d[:7]:>8s}' for d in DIMS) + '  rationale')
    for e in evalset:
        v = verdicts[e['id']]
        print(f'{e["id"]:2d} {e["kind"]:12s}' + ''.join(f'{v[d]:8d}' for d in DIMS) + f'  {v["rationale"][:90]}')
    print(f'\n{"kind":12s} {"n":>3s}' + ''.join(f'{d[:7]:>8s}' for d in DIMS))
    for kind in KINDS:
        ids = [e['id'] for e in evalset if kind is None or e['kind'] == kind]
        print(f'{kind or "all":12s} {len(ids):3d}' + ''.join(f'{np.mean([verdicts[i][d] for i in ids]):8.2f}' for d in DIMS))
    # Judge-reliability check: the judge and the model-free answer match should agree on clear cases.
    disagree = [e['id'] for e in evalset if (verdicts[e['id']]['correctness'] >= 4) != free[e['id']]['answer']]
    print('\njudge vs answer-match disagreements:', disagree or 'none')

t0 = time.monotonic()
verdicts = {e['id']: judge(e, results[e['id']]) for e in evalset}
print(f'judged {len(verdicts)} cases in {time.monotonic() - t0:.0f}s (cached verdicts are instant)\n')
print_judge(verdicts, free)

judged 15 cases in 0s (cached verdicts are instant)

id kind         correct groundi citatio  rationale
 1 rag                5       5       5  The assistant correctly identified the 'exp' claim and provided a precise, well-cited expl
 2 rag                5       5       5  The assistant correctly identified the status code and provided a citation that is support
 3 rag                5       5       5  The assistant correctly identifies the HttpOnly attribute and provides an explanation full
 4 rag                5       5       5  The assistant accurately reflects the RFC 8259 guidance while providing the necessary cont
 5 rag                5       5       5  The assistant accurately describes the required header and scheme as specified in the retr
 6 rag                5       5       5  The assistant correctly identifies that the seconds field can be 60 during a leap second, 
 7 rag                5       5       5  The assistant accurately identifies the Nil UUID value and its 

**Results** Every rag case retrieved its fact and answered correctly, and all three out-of-scope questions got the exact refusal after a search. The one miss is case 11 (compute): Claude searched three times, never called `run_python`, and refused, even though the two limits it needed were in its passages. The judge and the model-free answer match agree on all 15 cases.

**Judge caveat:** the judge gave 5/5/5 to every answer the string match accepted and 1/1/1 to the one it rejected. On this set it confirms the string match but adds no finer distinctions: grounding never separated a correct, supported answer from a correct, unsupported one. Testing that dimension needs harder cases whose answers are only partly supported by the passages.

## Part 3: Failure analysis
**The failure:** case 11, *"At minimum, how many bytes of cookies for a single domain should a general-use user agent be able to store?"* The answer, 204,800 (50 cookies × 4096 bytes), is not stated in RFC 6265. It has to be derived from two separate bullets in Section 6.1. Under `system_v1` the Part 2 run refused. The trace checks each component in turn: retrieval (did the facts reach the model?), routing (did it call the calculator?), and the prompt (does changing only the abstention rule change the behaviour?).

In [12]:
case = next(e for e in evalset if e['id'] == 11)
show_run(results[11])
print('both required facts present in the passages each search returned:')
for c in results[11]['calls']:
    if c['tool'] == 'search_rfcs':
        print(f"  {facts_in(case, c['result']['output'])!s:5s}  {c['args']['query']}")

> At minimum, how many bytes of cookies for a single domain should a general-use user agent be able to store?
  [OK ] turn 0: search_rfcs({"query": "minimum number of bytes per cookie user agent should support"}) -> 3 passages from RFC [6265]
  [OK ] turn 0: search_rfcs({"query": "cookie storage limits per domain user agent should be able to store at least"}) -> 3 passages from RFC [6265]
  [OK ] turn 1: search_rfcs({"query": "total cookie size in bytes allowed for a domain or per domain storage minimum"}) -> 3 passages from RFC [6265]
  answer: This is not covered by the indexed RFCs. 

both required facts present in the passages each search returned:
  False  minimum number of bytes per cookie user agent should support
  False  cookie storage limits per domain user agent should be able to store at least
  True   total cookie size in bytes allowed for a domain or per domain storage minimum


**Ablation 1: retrieval.** The user's question and each query Claude wrote, run through all four retrieval configurations. Each cell shows whether both required facts are in the top 3.

In [13]:
queries = [('user question', case['q'])] + [(f'claude query {i}', c['args']['query'])
           for i, c in enumerate((c for c in results[11]['calls'] if c['tool'] == 'search_rfcs'), 1)]
conds = [(ch, rr) for ch in CHUNKS for rr in (False, True)]
print(f'{"":15s}' + ''.join(f'{ch + (" +rr" if rr else ""):>15s}' for ch, rr in conds))
for label, q in queries:
    print(f'{label:15s}' + ''.join(f'{str(facts_in(case, [{"text": t} for _, t in retrieve(q, ch, rr)])):>15s}'
                                   for ch, rr in conds))

                     paragraph  paragraph +rr       sections   sections +rr


user question            False          False          False          False


claude query 1           False          False          False          False


claude query 2           False          False          False          False


claude query 3            True           True           True           True


**Ablation 2: the prompt.** The same question, five independent runs under each prompt version (trial 0 is the Part 2 run). `system_v2.txt` changes only rule 4.

In [14]:
TRIALS = 5
print(f'{"prompt":7s} {"refused":>8s} {"run_python":>11s} {"correct":>8s}')
for version in ('v1', 'v2'):
    rs = [ask(case['q'], version, trial=t) for t in range(TRIALS)]
    print(f'{version:7s} {sum(REFUSAL in r["answer"] for r in rs):6d}/{TRIALS} '
          f'{sum("run_python" in tools_called(r) for r in rs):9d}/{TRIALS} '
          f'{sum(free_metrics(case, r)["answer"] for r in rs):6d}/{TRIALS}')
print('\nv2 example:'); show_run(ask(case['q'], 'v2'))

prompt   refused  run_python  correct
v1           1/5         2/5      2/5
v2           0/5         5/5      5/5

v2 example:
> At minimum, how many bytes of cookies for a single domain should a general-use user agent be able to store?
  [OK ] turn 0: search_rfcs({"query": "minimum number of cookies per domain user agent should be able to store, minimum size of cookie in bytes"}) -> 3 passages from RFC [6265]
  [OK ] turn 1: run_python({"expression": "50 * 4096"}) -> 204800
  answer: RFC 6265 does not state a total byte minimum per domain. It says general-use user agents SHOULD support at least 50 cookies per domain and at least 4096 bytes per cookie [RFC 6265]. Taken together, that implies a capacity of 50 × 4096 = 204,800 bytes per domain, though this product is my derivation rather than a figure the RFC gives. 



**What the evidence shows**
- **Retrieval contributes but is not the cause.** RFC 6265 lists the two limits as separate one-line bullets, so each is its own chunk. Neither the user's question nor Claude's first two queries put both bullets in one top 3 under any of the four configurations, which is why the Part 2 run needed three searches. But the third search returned both facts together and the model still refused. In trials 1 to 4, Claude's first query retrieved both facts in a single search.
- **The break is in routing and generation.** Across five v1 runs, the model refused once, listed both limits but declined to multiply them twice, and computed 204,800 twice. Every v1 answer opens by noting that the RFC "does not give a per-domain byte total". The model reads rule 4 ("if the passages do not contain the answer... refuse") literally: a derived figure is not *contained* in the passages, so rule 4 competes with rule 3 ("use run_python for any arithmetic").
- **The prompt ablation isolates it.** `system_v2.txt` changes only rule 4: a derivable answer should be computed, and refusal is reserved for facts that are absent. Under v2 the model refused 0/5 times, called `run_python` 5/5 times, and answered correctly 5/5 times. Each answer still says the figure is derived rather than stated, which is the honest framing.

**Mitigation:** the `system_v2` rule 4 modification. An immediately obvious risk is that loosening this rule lets the model answer out-of-scope questions from its own knowledge. The cells below re-run all 15 cases under v2 as a regression check.

In [15]:
results_v2 = {e['id']: ask(e['q'], 'v2') for e in evalset}
free_v2 = {e['id']: free_metrics(e, results_v2[e['id']]) for e in evalset}
print_free(results_v2, free_v2)

id kind         tools called             tools  retr answer calls
 1 rag          search_rfcs                 ok    ok     ok     1
 2 rag          search_rfcs                 ok    ok     ok     2
 3 rag          search_rfcs                 ok    ok     ok     1
 4 rag          search_rfcs                 ok    ok     ok     2
 5 rag          search_rfcs                 ok    ok     ok     1
 6 rag          search_rfcs                 ok    ok     ok     1
 7 rag          search_rfcs                 ok    ok     ok     1
 8 rag          search_rfcs                 ok    ok     ok     1
 9 compute      run_python,search_rfcs      ok    ok     ok     3
10 compute      run_python,search_rfcs      ok    ok     ok     2
11 compute      run_python,search_rfcs      ok    ok     ok     2
12 compute      run_python,search_rfcs      ok    ok     ok     2
13 out-of-scope search_rfcs                 ok     -     ok     1
14 out-of-scope search_rfcs                 ok     -     ok     1
15 out-of-

In [16]:
verdicts_v2 = {e['id']: judge(e, results_v2[e['id']]) for e in evalset}
print_judge(verdicts_v2, free_v2)

id kind         correct groundi citatio  rationale
 1 rag                5       5       5  The assistant correctly identified the 'exp' claim and provided a citation that matches th
 2 rag                5       5       5  The assistant correctly identified the status code and provided a clear explanation suppor
 3 rag                5       5       5  The assistant correctly identifies the HttpOnly attribute and provides a well-supported ex
 4 rag                5       5       5  The assistant accurately reflects the RFC content, provides correct citations, and include
 5 rag                5       5       5  The assistant accurately describes the required header and scheme as specified in the retr
 6 rag                5       5       5  The assistant correctly identifies that the seconds field can be 60 during a leap second, 
 7 rag                5       5       5  The assistant accurately identifies the Nil UUID value and its purpose while correctly cit
 8 rag                5  

**v2 regression check:** The mitigation improves results without catastrophic regression: 15/15 on every model-free metric, a judge score of 5.00 on all three dimensions, and the three out-of-scope questions are still refused.